# A panel CodeEditor with automatic new lines

*My current intuition is that the Viewer design pattern based on a param reactive 'variable network' with a panel output is exactly what we need.*

**After long and heavy reading here is the elegant CodeEditor widget with automatic new line added.**

## Solved 

Finally, let's include the CodeEditor widget as a method. 

In [ ]:
import panel as pn
import param

pn.extension('codeeditor')

# this code is based on these two tutorials: 
# https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers
# https://panel.holoviz.org/reference/widgets/CodeEditor.html

class Labelizer(param.Parameterized):
    lines = param.String(default='# Hi there') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'

    def editor_widget(self): 
        title = pn.pane.Str('Annotations')
        editor = pn.widgets.CodeEditor.from_param(self.param.lines, max_height=100)
        return pn.Column(title, editor)
    

labelz = Labelizer()


In [ ]:
labelz.editor_widget()

# Wanderings

## Learning about CodeEditor to create a multi line text editor

Here is the  documentation: https://panel.holoviz.org/reference/widgets/CodeEditor.html

Here is some working code that I will adapt to learn better how it works. 

### A viewer based example (working, but without automatic newlines)

In [ ]:
import panel as pn
import param
from panel.viewable import Viewer
import re 

pn.extension('codeeditor')

class CodeEditorTest(Viewer):
    
    lines = param.String('''Hello World\n\n\n''')
    
    def __panel__(self):
        """ Map the string to appear as an Ace editor. """
        return pn.Param(
            self.param,
            widgets=dict(
                lines=dict(
                    type=pn.widgets.CodeEditor,
                    language='toml',
                )
            )
        )
    
edit = CodeEditorTest()

edit

It is possible to adjust the string value of the widget like this. 

In [ ]:
edit.editor_strin

### Simple widget (without reactivity) 

I would like to have more control on the actual widget. Let's try a more explicit code:

In [ ]:
editor_widget = pn.widgets.CodeEditor(value='#hi\n\n\n\n\n\n\n', max_height=150) 

In [ ]:
editor_widget

### Parametrized string with bidirectional reactivity

This code is explicit non-parametrized code. Now let's add some reactivity. 

In [ ]:
class TOMLText(param.Parameterized):
    lines = param.String(default='# Hi there\n') 

toml_txt = TOMLTxt()

editor_widget = pn.widgets.CodeEditor(value=toml_txt.param.lines, on_keyup=True, max_height=150) # reactive! 


In [ ]:
editor_widget

In [ ]:
editor_widget.value = 'A\nB\n'

In [ ]:
editor_widget.max_height = 50

## A `.param.watcher()` based automatic newlines (nice!)

This morning I have studied the [Dependencies and Watchers](https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#watchers) tutorial. That should fix our auto newline problem.  

In [ ]:
class Labelizer(param.Parameterized):
    lines = param.String(default='# Hi there') 

    def __init__(self, **params): 
        super().__init__(**params) 
        self.param.watch(self.add_newline, ['lines'], queued=True)

    def add_newline(self, event):  
        if (not self.lines.endswith('\n')): 
            self.lines = f'{self.lines}\n'
    

txt = Labelizer()

editor = pn.widgets.CodeEditor.from_param(txt.param.lines)

In [ ]:
editor

In [ ]:
txt.lines

### Towards automatic newlines (not working automatically and no bidirectional reactivity) 

Ok, this works. Now I would like to automatically include a newline at the end of the string. Le's try to add this as a method in the parametrized class.

In [ ]:
class TOMLTxt(param.Parameterized): 
    
    txt = param.String(default='# Hi there', allow_refs=True) 
    
    @param.depends('txt', watch=False) 
    def auto_newline(self): 
        if ~pn.rx(self.txt).endswith('\n'): 
            self.txt = f'{self.txt}\n'

toml_txt = TOMLTxt()

editor_widget = pn.widgets.CodeEditor(value=toml_txt.param.txt, max_height=150) # reactive! 


In [ ]:
editor_widget

In [ ]:
toml_txt.txt

In [ ]:
toml_txt.txt = 'Bye\n'

In [ ]:
pn.rx(toml_txt.txt).endswith('\n')

In [ ]:
toml_txt.txt = 'Bye\nBye'

In [ ]:
pn.rx(toml_txt.txt).endswith('\n')

In [ ]:
toml_txt.auto_newline()

By switching to watch=False I no longer get a recursion error. Somehow I am still missing the logic here. Also, when editing the values the widget, they do not get updated in the parametrized instance. 

### What if we (auto) return self.txt?  (no bidirectionality) 

In [ ]:
class TOMLTxt(param.Parameterized): 
    
    txt = param.String(default='# Hi there', allow_refs=True) 
    
    @param.depends('txt', watch=False) 
    def auto_newline(self): 
        if ~pn.rx(self.txt).endswith('\n'): 
            self.txt = f'{self.txt}\n'
        return self.param.txt

toml_txt = TOMLTxt()

editor_widget = pn.widgets.CodeEditor(value=toml_txt.auto_newline(), max_height=150) # reactive! 


In [ ]:
editor_widget

In [ ]:
editor_widget.param.value.rx()

### Nice! Bi-directional with .fromparam() works!

In [ ]:
class TOMLTxt(param.Parameterized):
    txt = param.String(default='# Hi there') 

toml_txt = TOMLTxt()

editor_fromparam = pn.widgets.CodeEditor.from_param(toml_txt.param.txt, max_height=50)

In [ ]:
editor_fromparam

In [ ]:
toml_txt.txt

### Bound function (not working)

Ok, now we need to create the auto newline functionality. Perhaps with a bound function first? 

In [ ]:
def add_newline(txt): 

    if not txt.endswith('\n'): 
        txt = f'{txt}\n'
    return txt 
    

In [ ]:
bound_add_newline = pn.bind(add_newline, toml_txt.param.txt)

In [ ]:
class TOMLTxt(param.Parameterized):
    txt = param.String(default='# Hi there') 

toml_txt = TOMLTxt()

#bound_editor_fromparam = pn.widgets.CodeEditor.from_param(bound_add_newline.rx(), max_height=50)

### Power up the TOMLTxt class (not working) 

Let's try to power up the TOMLTxt class somehow. The problem seems somehow similar to the GoogleMaps example here: https://holoviz-dev.github.io/panel/tutorials/intermediate/reusable_components.html#allow-references 

Or perhaps similar to this TextFormatter: https://panel.holoviz.org/explanation/api/param.html#references

In [ ]:
class TOMLTxt(param.Parameterized):
    txt = param.String(default='# Hi there', allow_refs=True) 
    txt_with = param.String(allow_refs=True) 
    
    # this allows for instantation with custom values 
    def __init(self, **params): 
        super().__init__(self,**params)

        # this does not seem to do a lot 
        self.txt_with = pn.rx(f'{self.param.txt}\n')
        

toml_txt = TOMLTxt(txt='ABC')
editor_fromparam = pn.widgets.CodeEditor.from_param(toml_txt.param.txt, max_height=50)

In [ ]:
editor_fromparam

In [ ]:
toml_txt.txt

Perhaps the answer is to include a callback method that returns a CodeEditor widget?

### Nested dependency?

Are we dealing with nested dependencies here: See: https://param.holoviz.org/en/docs/latest/user_guide/Dependencies_and_Watchers.html#dependency-specs

In [ ]:
pn.widgets.CodeEditor(value=pn.rx(obj=toml_txt.param.txt))

In [ ]:
toml_txt.param.txt.rx()

## Fighting with a multi line text input (can't enter Enter)

See: https://holoviz-dev.github.io/panel/how_to/custom_components/examples/plot_viewer.html

In [ ]:
import param 
import panel as pn
from panel.viewable import Viewer
import toml 

pn.extension(template='fast')
pn.extension('texteditor')

In [ ]:
text_area_input = pn.widgets.TextAreaInput(label='Text Area Input', placeholder='Enter a string here...')
text_area_input

In [ ]:
class CubeViewer(Viewer): 

    toml_txt = param.String(default='''hi:''')

    def __panel__(self): 

        text_widget = pn.widgets.TextAreaInput(name='TOML', auto_grow=True).from_param(self.param.toml_txt)

        return pn.Column(text_widget)
       

In [ ]:
cube = CubeViewer()

In [ ]:
cube

In [ ]:
cube.toml_txt = 'bye'

We see that basically this works. Let's try to extend it with a more fancy panel widget, and then see  how we can include the toml_txt as a keyword argument in the instantiation.

In [ ]:
pn.widgets.TextAreaInput(name='Growing TextArea', auto_grow=True, rows=10, max_rows=20, value="""\
This text area will grow when newlines are added to the text:

1. Foo
2. Bar
3. Baz 





""", width=500)

In [ ]:
wysiwyg = pn.widgets.TextEditor(placeholder='Enter some text')
wysiwyg